**Human Movement Intelligence | Biomechanical Load Estimation | Presented by: Reza Saadatyar (2026)**

**03 | Numerical Signal Recovery and Temporal Alignment Audit**

**Research objective:** Determine whether published MATLAB v5 object representations expose verifiable numerical IMU/FE arrays through the approved Python toolchain, and identify the remaining evidence needed to construct aligned, participant-linked femoral-neck strain trajectories.

**Primary dataset:** Hollinger et al. (2026), *Scientific Data*, [10.1038/s41597-026-07429-3](https://doi.org/10.1038/s41597-026-07429-3). Exact Figshare dataset: [30225739, version 13](https://doi.org/10.6084/m9.figshare.30225739.v13). Source consists of original processed participant MATLAB files and a separately published participant workbook.

**Research inputs:** The approved Stage 02 record registry and provenance JSON, and original source MAT files reverified against the pinned archive checksum. The 444-record BoneNet derivative is **not** substituted for the original six-activity dataset.

**Scope:** Source-registry integrity, activity-specific FE target coverage, a bounded representative inspection of nested MATLAB representations, numerical-leaf candidate reporting, temporal-axis feasibility and explicit scientific Quality Gates. Any numeric leaf found inside an opaque MATLAB representation remains **unattributed** until its class, original channel/target name, physical units and indexing are independently confirmed.

**Key methodological boundary:** The published IMU trajectories are time-normalised; the article describes FE evaluation at every tenth point. Candidate arrays with axes of length 101 or 11 cannot be assumed to represent calibrated IMU or independently computed FE strain without additional validation. Never interpolate or treat FE-derived targets as direct in-vivo measurements.

**Expected decision:** Either identify a verifiable numeric conversion route with independently checked semantics, or record that an original MATLAB-native conversion step is required. No regression, model fitting, patient-level inference or external validation is performed.

**Visual workflow:** The final scientific SVG will be inserted after the executed results and their interpretation have been approved. Its visible title will be scientific only, without stage or notebook numbering.

**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.split("/")[-1].removesuffix(".git")

%cd /content

if not repo_path.exists():
    !git clone -q {repo_url}


# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}

**01. Imports and Environment**

In [ ]:
# ======================== 01. Imports and Environment ========================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("biomechanical_load_estimation")


# ---------------------------- 2. Standard Library ----------------------------
import json
from pathlib import Path


# ----------------- 3. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd
import scipy
from scipy.io import whosmat
from scipy.io.matlab import matfile_version


# ------------- 4. Scientific Visualisation and Notebook Display --------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

plot_style(font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11, legend_size=10,
    figure_dpi=100, figure_size=(9, 4))


# -------------- 5. Source Integrity and Data Recovery Utilities --------------
from core.output_download import download_outputs_from_colab
from essex_figshare import (
    ARTICLE_ID, ARTICLE_VERSION, fetch_version_manifest,
    find_published_file, ensure_source_file, inspect_matlab_archive,
    extract_matlab_files,
)
from essex_numeric_recovery import (
    check_record_registry, select_diagnostic_records,
    inspect_sampled_records, candidate_inventory,
    activity_fe_coverage, assert_source_fe_status_matches,
)


# ---------------------- 6. Source and Output Locations -----------------------
study_dir = root / "biomechanical_load_estimation"
raw_dir = root / "data" / "raw" / "essex_femoral_neck_loading"
mat_dir = raw_dir / "matlab_records"
output_dir = study_dir / "output"
raw_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

print("Dataset:", ARTICLE_ID, "| version:", ARTICLE_VERSION)
print("SciPy:", scipy.__version__)
print("Research outputs:", output_dir)

**02. Restore the Pinned Source and Verify Its Integrity**

The raw Essex MATLAB archive is intentionally excluded from Git. Colab sessions may start without local data, so this notebook reuses the official Figshare v13 acquisition utility and compares the downloaded archive's SHA256 with the Stage 02 provenance record.

A matching filename or number of participant files is insufficient evidence of source identity. The archive contents, recorded digest and the presence of the exact original participant files must agree before new analysis. The original workbook and protected Stage 01/02 results remain unchanged.

In [ ]:
# =========== 02. Versioned Source Archive and Evidence Validation ============

# ------------------- 1. Load Historical Research Evidence --------------------
provenance_path = output_dir / "02_record_provenance_and_quality_gate.json"
registry_path = output_dir / "02_original_record_registry.csv"

if not provenance_path.is_file() or not registry_path.is_file():
    raise FileNotFoundError(
        "The tracked Stage 02 provenance and registry are required."
    )

stage02 = json.loads(provenance_path.read_text(encoding="utf-8"))
if (
    stage02["source_article_id"] != ARTICLE_ID
    or stage02["version"] != ARTICLE_VERSION
    or stage02["source_registry_gate"] != "PASSED"
):
    raise ValueError("Stage 02 does not attest to the required Figshare source.")


# ------------ 2. Retrieve and Verify the Published MATLAB Source -------------
article, available_files = fetch_version_manifest()
archive_item = find_published_file(available_files, "MATLAB.zip")
zip_path, zip_hashes = ensure_source_file(archive_item, raw_dir)
if zip_hashes["sha256"] != stage02["archive_sha256"]:
    raise ValueError("Original archive fingerprint differs from Stage 02.")

zip_inventory = inspect_matlab_archive(zip_path, expected_mat_files=40)
mat_paths = extract_matlab_files(zip_path, mat_dir, zip_inventory)
assert len(mat_paths) == 40
print("Source integrity: VERIFIED against Stage 02")
print("MATLAB participant files:", len(mat_paths))
print("Source SHA256:", zip_hashes["sha256"])

**03. Source-Record Identity and Cohort Contract**

The analytical unit for this investigation is a recorded movement element with an explicit original participant and updated trial identifier. We use the full source `local_record_key` to ensure every inspected payload can be traced to a specific published MAT file and record index. `UpdatedTrialID` is readable across the audited set, but it is not yet evidence that the FE targets have been numerically decoded and correctly aligned.

The complete Stage 02 source registry is treated as the approved starting evidence. No randomly generated trial IDs, inferred subject order or training splits are introduced.

In [ ]:
# ============ 03. Historical Record Registry and Source Identity =============

# ---------------- 1. Read and Verify the Saved Trial Registry ----------------
registry = pd.read_csv(
    registry_path,
    dtype={
        "subject_id": "string",
        "source_file": "string",
        "local_record_key": "string",
        "task": "string",
        "updated_trial_id": "string",
        "fe_representation": "string",
        "imu_representation": "string",
    },
)
identity_summary = check_record_registry(registry, stage02)
assert_source_fe_status_matches(registry, stage02)

file_ids = {path.stem.lower() for path in mat_paths}
assert file_ids == set(registry["subject_id"])
print("Historical source records:", identity_summary["records"])
print("Distinct participants:", identity_summary["participants"])
print("Readable source identifiers and registry: VERIFIED")


# ----------------- 2. Save a Compact Traceable Audit Summary -----------------
registry_summary = pd.DataFrame([identity_summary])
registry_summary.to_csv(
    output_dir / "03_registry_integrity_summary.csv", index=False
)
display(registry_summary)

**04. Original Activities and Finite-Element Target Availability**

The source `Data` arrays cover multiple movement activities, subconditions and trials. Before interpreting waveforms, we must understand where FE values are structurally available and where they are missing.

This descriptive audit cross-tabulates original `Task` categories and `e1e3` representation states. Nonempty MATLAB-object fields are **potentially present** FE outputs, not verified numerical strain trajectories. The excluded participants and partial FE gaps are retained in the accounting rather than silently dropped.

In [ ]:
# ============= 04. Movement Activity and FE Availability Profile =============

# -------------- 1. Summarise Original Activity and FE Coverage ---------------
activity_counts = (
    registry.groupby("task").size().rename("record_count")
    .reset_index().sort_values("record_count", ascending=False)
)
fe_coverage = activity_fe_coverage(registry)
fe_table = fe_coverage.pivot(
    index="task", columns="fe_representation", values="record_count"
).fillna(0).astype(int)

display(activity_counts)
display(fe_table)


# --------------- 2. Audit Partial and Participant-Wide FE Gaps ---------------
subject_coverage = (
    registry.groupby(["subject_id", "fe_representation"])
    .size().rename("record_count").reset_index()
)
fe_coverage.to_csv(output_dir / "03_activity_fe_coverage.csv", index=False)
subject_coverage.to_csv(
    output_dir / "03_participant_fe_coverage.csv", index=False
)
print("Original activities:", registry["task"].nunique())
print("FE fields missing by representation:",
      int((registry["fe_representation"] == "empty").sum()))

**05. MATLAB v5 Container and Object-Encoding Inspection**

The original MAT files contain MATLAB-serialised objects, including variables that SciPy may represent as `MatlabOpaque`. This representation does **not** establish that Python has recovered the underlying table values. We first inspect the MAT container headers without exporting hidden workspace bytes.

SciPy's documented MATLAB-object limitations mean that a structurally readable `Data` field is insufficient for extracting correctly labelled IMU channels or FE targets. See [SciPy issue 22736](https://github.com/scipy/scipy/issues/22736) and the [`loadmat` documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.io.loadmat.html).

In [ ]:
# ============= 05. MAT Container and Internal Workspace Headers ==============

# -------------- 1. Inspect Representative Original MAT Headers ---------------
container_rows = []
for source in mat_paths:
    major, minor = matfile_version(source)
    for variable_name, shape, matlab_class in whosmat(source):
        container_rows.append({
            "source_file": source.name,
            "mat_major": int(major),
            "mat_minor": int(minor),
            "source_variable": variable_name,
            "source_shape": str(tuple(shape)),
            "matlab_class": matlab_class,
        })
containers = pd.DataFrame(container_rows)
display(containers.head(12))
containers.to_csv(
    output_dir / "03_matlab_container_inventory.csv", index=False
)
print("MAT container variables recorded:", len(containers))
print("Files with MATLAB workspace reference:",
      containers.loc[
          containers["source_variable"] == "__function_workspace__",
          "source_file"
      ].nunique())

**06. Bounded Recovery Probe of Original IMU and FE Objects**

This diagnostic is deliberately limited to representative original source elements selected without model outcomes. It includes at least one record per participant and activity wherever the corresponding records exist, and cases with empty or nonempty FE fields. No entire object is printed as a large, unstructured Python representation.

The study-specific utility walks MATLAB structs and nested arrays under strict depth, node and child limits. It records **paths, types, shapes and possible numeric leaves**, but not raw participant signals. A numeric leaf discovered beneath an opaque MATLAB wrapper is an **unverified candidate**: its presence alone does not identify a sensor channel, FE target, time grid or physical unit.

In [ ]:
# ============ 06. Bounded Original MATLAB Numeric Path Inspection ============

# ----------- 1. Select Representative Published Record Identifiers -----------
selected_records = select_diagnostic_records(
    registry, max_samples=64
)
display(selected_records[[
    "subject_id", "task", "updated_trial_id",
    "local_record_key", "fe_representation"
]].head(16))
print("Source records selected for bounded introspection:",
      len(selected_records))


# --------- 2. Inspect Nested MATLAB Values without Implicit Decoding ---------
payload_paths, probe_summary = inspect_sampled_records(
    mat_dir, selected_records, max_depth=9, max_nodes=300
)
assert len(probe_summary) == len(selected_records) * 2

payload_paths.to_csv(
    output_dir / "03_matlab_object_path_inventory.csv", index=False
)
probe_summary.to_csv(
    output_dir / "03_bounded_probe_registry.csv", index=False
)
display(probe_summary.groupby("modality")[
    ["nodes_inspected", "numeric_leaves_visible", "inspection_truncated"]
].sum())
print("Truncated source-object probes:",
      int(probe_summary["inspection_truncated"].sum()))

**07. Numerical Candidates, FE Time Grid and Alignment Evidence**

For each physically meaningful waveform we would need: a verified original MATLAB variable or table column, a numeric data array, correct channel and axis labels, finite values, documented units and a deterministic mapping to a participant, activity, intensity and trial.

The published dataset describes 101-point normalised movement cycles and less densely computed FE estimates. The following **shape-screening step is exploratory only**: an axis of length 101 or 11 does not establish the correct time convention. We do not convert physical units, generate interpolated FE values or create model-ready arrays based solely on shape.

In [ ]:
# ========== 07. Numeric Leaf Candidates and Temporal Axis Evidence ===========

# ----------- 1. Preserve All Numeric Leaf Candidates as Unverified -----------
candidate_table = candidate_inventory(payload_paths)
candidate_table.to_csv(
    output_dir / "03_numeric_leaf_candidates.csv", index=False
)
print("Numeric leaves discovered:", len(candidate_table))
print("Shape-compatible with 101 or 11 positions:",
      int(candidate_table["time_axis_candidate"].sum()) if len(candidate_table) else 0)

if candidate_table.empty:
    print("No attributable numerical IMU/FE arrays were recovered in sampled objects.")
else:
    display(candidate_table[[
        "subject_id", "modality", "path", "shape", "dtype",
        "beneath_matlab_object", "time_axis_candidate",
        "usable_for_prediction",
    ]].head(25))


# --------- 2. Record Actual Candidate Shapes without Assuming Units ----------
time_candidates = candidate_table.loc[
    candidate_table["time_axis_candidate"]
].copy()
time_candidates.to_csv(
    output_dir / "03_temporal_axis_candidates.csv", index=False
)
print("Target axis correspondence to published FE computation: UNVERIFIED")
print("Original MATLAB sensor / target column names: NOT YET VALIDATED")
print("Prediction-ready numeric tensor constructed: NO")

**08. Data Coverage and Recovery Diagnostic Figure**

The first panel shows the actual original source activity distribution, separated by FE field representation. The second panel shows the number of numeric leaves observed in the bounded IMU and FE object probes. These results are **diagnostic counts**, not proof that any numerical leaf represents a valid physical trajectory.

The plot uses the repository's canonical scientific style and is saved as a traceable stage-specific PNG. No artificial sine waves, simulated IMU signals or fabricated FE values are plotted.

In [ ]:
# ========= 08. Activity Coverage and Numeric Recovery Visualisation ==========

# ------------- 1. Visualise FE Source Coverage across Activities -------------
fig, axes = plt.subplots(2, 1, figsize=(10, 8), constrained_layout=True)
fe_table.sort_index().plot(
    kind="bar", stacked=True, ax=axes[0], rot=25, width=0.78
)
axes[0].set_title("Original Essex movement records by FE representation")
axes[0].set_ylabel("Recorded movement elements")
axes[0].set_xlabel("Source task")
axes[0].legend(title="FE source representation")


# ------------- 2. Visualise Bounded Numerical Recovery Evidence --------------
numeric_probe_counts = (
    probe_summary.groupby("modality")["numeric_leaves_visible"].sum()
)
axes[1].bar(
    numeric_probe_counts.index, numeric_probe_counts.values
)
axes[1].set_title("Numerical leaves visible in bounded MATLAB probes")
axes[1].set_ylabel("Unverified numeric leaves")
axes[1].set_xlabel("Original source field")
axes[1].set_ylim(bottom=0)

save_figure(fig, output_dir / "03_source_recovery_diagnostics.png")
plt.show()

**09. Scientific Quality Gate and Next-Step Decision**

The *data-integrity gate* requires that the archived source version and checksum match Stage 02, that all 40 original files are present, that the approved registry has 1,897 unique record keys and that the original FE availability counts reconcile. The *numeric-conversion readiness gate* is separate: it requires verified numerical arrays, schema-labelled channels, physical units and time-grid alignment.

The current investigation may pass the structural integrity check while the model-readiness gate remains blocked. A MATLAB-native export or independently demonstrated object-decoding procedure is then a scientifically appropriate next step. A readable candidate shape alone cannot unblock supervised regression.

In [ ]:
# ========= 09. Source Integrity and Numerical Readiness Quality Gate =========

# ------------ 1. Validate Independent Input and Probe Accounting -------------
assert zip_hashes["sha256"] == stage02["archive_sha256"]
assert len(mat_paths) == 40
assert identity_summary["records"] == 1897
assert identity_summary["participants"] == 40
assert int(fe_coverage["record_count"].sum()) == len(registry)
assert probe_summary["local_record_key"].nunique() == len(selected_records)
assert int(probe_summary.groupby("modality").size().sum()) == (
    2 * len(selected_records)
)
assert not candidate_table.get(
    "usable_for_prediction", pd.Series([], dtype=bool)
).any()


# ------------ 2. Save Conservative Scientific Readiness Decision -------------
quality = {
    "figshare_article": ARTICLE_ID,
    "dataset_version": ARTICLE_VERSION,
    "archive_sha256": zip_hashes["sha256"],
    "stage02_provenance_file": provenance_path.name,
    "participants": identity_summary["participants"],
    "source_records": identity_summary["records"],
    "activities_observed": sorted(registry["task"].dropna().unique().tolist()),
    "sampled_original_records": int(len(selected_records)),
    "bounded_object_probes": int(len(probe_summary)),
    "truncated_probes": int(probe_summary["inspection_truncated"].sum()),
    "unverified_numeric_leaf_candidates": int(len(candidate_table)),
    "101_or_11_axis_candidates": int(len(time_candidates)),
    "data_integrity_gate": "PASSED",
    "verified_numeric_imu_channels": False,
    "verified_numeric_fe_targets": False,
    "verified_fe_target_units": False,
    "verified_imu_fe_temporal_alignment": False,
    "numerical_model_readiness": False,
    "next_step": (
        "Independent MATLAB-object decoding and native-table schema verification; "
        "do not construct supervised regression tensors yet"
    ),
    "training_or_independent_test_performed": False,
}
(output_dir / "03_numeric_recovery_quality_gate.json").write_text(
    json.dumps(quality, indent=2) + "\n", encoding="utf-8"
)
display(pd.DataFrame([{
    "source_integrity": quality["data_integrity_gate"],
    "sampled_records": quality["sampled_original_records"],
    "numeric_candidates": quality["unverified_numeric_leaf_candidates"],
    "model_ready": quality["numerical_model_readiness"],
}]))
print("SOURCE DATA INTEGRITY: PASSED")
print("NUMERICAL MODEL READINESS: NOT VERIFIED")
print("Regression training or independent evaluation: NOT PERFORMED")

**10. Research Evidence Export and Scientific Interpretation**

Save each stage-specific table and the PNG figure with the `03_` prefix. The output ZIP is prepared for manual retrieval from Colab, not committed automatically. Source MATLAB files remain in `data/raw/` and must not be pushed to a public repository.

**Interpretation boundary:** After execution, report how many numeric candidates were actually found, whether the bounded inspection was truncated, and what the data reveal about MATLAB-object support. Do not describe the project as ready for regression until the underlying sensor and FE values are independently verified. The final stage-specific scientific SVG will be created from the actual recorded outputs after review and will **not** display a stage or notebook number in its title.

In [ ]:
# ============= 10. Stage-Specific Output Verification and Export =============

# --------------- 1. Verify the Complete Research Evidence Set ----------------
required = [
    "03_registry_integrity_summary.csv",
    "03_activity_fe_coverage.csv",
    "03_participant_fe_coverage.csv",
    "03_matlab_container_inventory.csv",
    "03_matlab_object_path_inventory.csv",
    "03_bounded_probe_registry.csv",
    "03_numeric_leaf_candidates.csv",
    "03_temporal_axis_candidates.csv",
    "03_source_recovery_diagnostics.png",
    "03_numeric_recovery_quality_gate.json",
]
missing = [name for name in required if not (output_dir / name).is_file()]
if missing:
    raise FileNotFoundError(
        "Required research evidence missing: " + ", ".join(missing)
    )
print("Recorded Stage 03 files:", len(required))


# ------------- 2. Package Colab Evidence for Independent Review --------------
download_outputs_from_colab(
    folder="biomechanical_load_estimation/output", stage="03"
)